# Obtaining FASTA files from Entrez Database
The BioPython library provides a function to query the NCBI database for specific species and genes of interest.

In [23]:
from Bio import Entrez, SeqIO # Get this library from `pip install biopython`
from io import StringIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import time
import pandas as pd
from typing import List
import os

Get the bacteria needed from bacteria_metadata.csv

In [24]:
# Read species list from an Excel file without headers (first column)
species_df = pd.read_excel("bla remaining.xlsx", header=None)
species_list = species_df[0].tolist()

In [25]:
# Genes of interest
genes = ["bla"]

Entrez.email = "acqi@andrew.cmu.edu"

In [26]:
def QueryEntrezDatabase(species_list: List[str], gene_list: List[str], output_folder="./"):
    """ Main function to query from NCBI database to obtain genes across bacterial species. Note this only searches for the "gene" terms, not rRNA.

    Args:
        species_list (List[str]): List of strings representing bacterial species names
        gene_list (List[str]): List of genes you want to get from the database
        output_folder (str, optional): Defaults to "./". Specify the folder/directory you want the sequences to be saved to
    """
    
    # Save each gene to its own fasta file, iterate over genes first then species next
    for gene in gene_list:
        
        # Name the file based on the gene name
        output_filename = output_folder + f"{gene}_cds_all_bacteria.fasta"
        
        for species in species_list:
            
            # We want only the DNA sequence of the full CDS of this gene across species
            database = "nucleotide"
            search_term = f"{species}[Organism] AND {gene}[Gene Name] AND CDS[Feature] NOT UNVERIFIED[Text]"
            retmax_search = "50" # Increase retmax to potentially find multiple CDS for a gene in a species
            rettype_fetch = "gb" # Fetch in GenBank format to get feature information
            retmode_fetch = "text"
            
            found_cds_for_species = False

            try:
                # Perform the search
                handle = Entrez.esearch(db=database, term=search_term, retmax=retmax_search)
                search_results = Entrez.read(handle)
                handle.close()
                id_list = search_results["IdList"]

                # If we have search results:
                if id_list:
                    print(f"Found {len(id_list)} records with IDs: {id_list}")

                    # Loop thru each record
                    for record_id in id_list:
                        
                        if not found_cds_for_species:
                            # Use Entrez query to the database
                            try:
                                handle = Entrez.efetch(db=database, id=record_id, rettype=rettype_fetch, retmode=retmode_fetch)
                                genbank_record = handle.read()
                                handle.close()

                                for seq_record in SeqIO.parse(StringIO(genbank_record), "genbank"):
                                    organism = seq_record.annotations.get("organism")
                                    
                                    # Ensure the organism in the record matches the queried species
                                    if organism == species: 
                                        
                                        for feature in seq_record.features:
                                            
                                            # Only get full CDS, not partial
                                            if feature.type == "CDS":
                                                if "partial" not in feature.qualifiers:
                                                    
                                                    gene_name_qualifier = feature.qualifiers.get('gene')
                                                    
                                                    if gene_name_qualifier and gene_name_qualifier[0] == gene:
                                                        try:
                                                            cds_seq = feature.extract(seq_record.seq)
                                                            fasta_description = organism
                                                            fasta_id = f"{organism.replace(' ', '_')}_{gene}"
                                                            fasta_record = SeqRecord(cds_seq, id=fasta_id, description=fasta_description)

                                                            with open(output_filename, "a") as outfile:
                                                                SeqIO.write(fasta_record, outfile, "fasta")
                                                            print(f"      - Extracted full CDS for '{gene}' in '{organism}' (Record ID: {seq_record.id}).")
                                                            found_cds_for_species = True
                                                            
                                                            # Break inner feature loop after finding one full CDS for the target gene
                                                            break 
                                                        except Exception as e:
                                                            pass # Don't log anything
                                                            #print(f"Error extracting CDS from record {seq_record.id}: {e}")
                                                else:
                                                    print(f"      - Skipping partial CDS from record {seq_record.id}")

                            except Exception as e:
                                print(f"Error fetching record with ID {record_id}: {e}")

                        # If we found the CDS, break the query
                        else:
                            #print(f"Full CDS already found for '{gene}' in '{species}'. Skipping remaining records.")
                            break
                else:
                    print(f"    No records found for '{gene}' in '{species}' with full CDS and excluding 'UNVERIFIED'.")

            except Exception as e:
                print(f"  An error occurred during search for '{gene}' in '{species}': {e}")

            time.sleep(1) # Wait 1 sec after each species query

        time.sleep(1) # Wait 1 secs before starting the next gene queries

def QueryEntrezDatabase_rRNA(species_list: List[str], gene_list: List[str], output_folder="./"):
    """Query NCBI to obtain rRNA sequences (e.g., 16s rRNA) across bacterial species.

    Args:
        species_list (List[str]): Bacterial species names.
        gene_list (List[str]): rRNA gene names to search.
        output_folder (str): Folder where output FASTA files will be saved.
    """
    
    for gene in gene_list:
        output_filename = os.path.join(output_folder, f"{gene}_rRNA_all_bacteria.fasta")
        
        for species in species_list:
            database = "nucleotide"
            # Broad search: use the gene term without restricting to CDS
            search_term = f"{species}[Organism] AND {gene}[All Fields] NOT UNVERIFIED[Text]"
            retmax_search = "50"
            rettype_fetch = "gb"
            retmode_fetch = "text"
            
            found_rRNA_for_species = False

            try:
                handle = Entrez.esearch(db=database, term=search_term, retmax=retmax_search)
                search_results = Entrez.read(handle)
                handle.close()
                id_list = search_results["IdList"]

                if id_list:
                    print(f"Found {len(id_list)} records for '{gene}' in '{species}'")
                    
                    for record_id in id_list:
                        if not found_rRNA_for_species:
                            try:
                                handle = Entrez.efetch(db=database, id=record_id, rettype=rettype_fetch, retmode=retmode_fetch)
                                genbank_record = handle.read()
                                handle.close()

                                for seq_record in SeqIO.parse(StringIO(genbank_record), "genbank"):
                                    organism = seq_record.annotations.get("organism")
                                    if organism == species:
                                        for feature in seq_record.features:
                                            if feature.type == "rRNA":
                                                product = feature.qualifiers.get("product", [""])[0]
                                                if "16s" in product.lower():
                                                    rRNA_seq = feature.extract(seq_record.seq)
                                                    fasta_id = f"{organism.replace(' ', '_')}_{gene}"
                                                    fasta_record = SeqRecord(rRNA_seq, id=fasta_id, description=organism)
                                                    with open(output_filename, "a") as outfile:
                                                        SeqIO.write(fasta_record, outfile, "fasta")
                                                    print(f"  - Extracted rRNA for '{gene}' in '{organism}' (Record ID: {seq_record.id}).")
                                                    found_rRNA_for_species = True
                                                    break
                                        if found_rRNA_for_species:
                                            break
                            except Exception as e:
                                print(f"Error fetching record with ID {record_id}: {e}")
                        else:
                            break
                else:
                    print(f"No records found for '{gene}' in '{species}'.")
            except Exception as e:
                print(f"An error occurred during search for '{gene}' in '{species}': {e}")

            time.sleep(1)
        time.sleep(1)

In [27]:
base_folder = "./Data/"
if not os.path.exists(base_folder):
    os.makedirs(base_folder)
    
QueryEntrezDatabase(species_list, genes, base_folder)
# QueryEntrezDatabase_rRNA(species_list, genes, base_folder)

    No records found for 'bla' in 'Rhodobacter sphaeroides' with full CDS and excluding 'UNVERIFIED'.
Found 50 records with IDs: ['2693714485', '2693637242', '2693715692', '2678048640', '2677982506', '2677910773', '2677852025', '2677548787', '2677548627', '2170081709', '1786242040', '1848409563', '2176969098', '1965354216', '2648900507', '2642083824', '2642081781', '2642076753', '2642075881', '2642047131', '2642046005', '2642065259', '2642064830', '2790584549', '2790582086', '2790571670', '2272061908', '2209274214', '2644598660', '2644597053', '2642088801', '2642082232', '2642074524', '2642044150', '2642041710', '2642089198', '2642087254', '2642082546', '2642049521', '2912114083', '2912105347', '2912077776', '2912067792', '1829034658', '1828945322', '2911538160', '2911518861', '2910328554', '1829037527', '1121641849']
Found 50 records with IDs: ['2573185702', '2667345567', '1129014675', '2890792873', '2087305806', '1878032874', '1797874700', '1333427791', '1217741302', '2862732542', '2